
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/molecular_docking_tutorial/blob/main/practical_docking/04_Kinase_Panel.ipynb)

# Kinase Panel — 다중 키나아제 선택성 프로파일링

## 목적

하나의 약물 후보가 **여러 키나아제 중 어떤 것에 선택적으로 결합**하는지 평가합니다.
선택성이 높을수록 부작용이 적은 약물이 될 가능성이 큽니다.

## 이론적 배경

### 키나아제 선택성 (Kinase Selectivity)
인간 게놈에는 ~500개의 키나아제가 있으며, ATP binding site의 구조가 유사합니다.
약물이 타겟 외의 키나아제에도 결합하면 off-target 독성이 발생할 수 있습니다.

### Cross-docking
동일한 리간드를 여러 키나아제 구조에 도킹하여 결합력을 비교합니다.
- 타겟 키나아제에 강하게 결합 + 다른 키나아제에 약하게 결합 → **선택적**
- 모든 키나아제에 비슷하게 결합 → **비선택적 (pan-kinase)**

### 구조 정렬
서로 다른 키나아제라도 ATP binding site 영역은 진화적으로 보존되어 있어,
Cα 원자 기준으로 구조를 정렬하고 포즈를 비교할 수 있습니다.

## 워크플로우
1. 키나아제 패널 정의 (PDB 코드 + 그룹)
2. 전체 구조 준비 & 정렬
3. Docking Box 설정 + 3D 확인
4. Cross-docking (각 키나아제 × 리간드)
5. 선택성 히트맵 & ProLIF 분석
6. 내보내기


## 0. 환경 설정


### 환경 설치

필요한 패키지를 설치합니다.


In [ ]:
#@title 환경 설치 {display-mode: "form"}
import subprocess, sys, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

pip_install('rdkit', 'gemmi', 'openbabel-wheel')
pip_install('pdbfixer', 'openmm')
pip_install('py3Dmol', 'prolif', 'MDAnalysis')
pip_install('seaborn', 'pandas', 'matplotlib', 'requests')
try: pip_install('pymol-open-source')
except: pass

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)
smina_path = os.path.join(BIN_DIR, 'smina')
if not os.path.exists(smina_path):
    import stat, urllib.request
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    os.chmod(smina_path, os.stat(smina_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('Done.')


### 라이브러리 로드


In [ ]:
#@title 라이브러리 로드 {display-mode: "form"}
import warnings; warnings.filterwarnings('ignore')
import os, subprocess, urllib.request, time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, SanitizeFlags
from openbabel import pybel
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import MDAnalysis as mda
from MDAnalysis.analysis import align as mda_align
import prolif as plf

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
WORK_DIR = '/content/kinase_panel' if os.path.exists('/content') else os.path.join(os.path.expanduser('~'), 'kinase_panel')
os.makedirs(WORK_DIR, exist_ok=True)
print('All libraries loaded.')


## 1. 키나아제 패널 & 리간드 정의


### 키나아제 패널 정의

비교할 키나아제를 PDB 코드와 함께 정의합니다.
기본 패널은 TK(타이로신 키나아제), CMGC, TKL 그룹에서 대표 5종을 포함합니다.
필요에 따라 추가/제거할 수 있습니다.


In [ ]:
#@title 1-1. 키나아제 패널 정의 {display-mode: "form"}

KINASES = [
    {'name': 'EGFR',  'pdb': '1M17', 'chain': 'A', 'group': 'TK',   'disease': 'NSCLC'},
    {'name': 'ABL1',  'pdb': '1IEP', 'chain': 'A', 'group': 'TK',   'disease': 'CML'},
    {'name': 'JAK2',  'pdb': '3FUP', 'chain': 'A', 'group': 'TK',   'disease': 'MPN'},
    {'name': 'CDK2',  'pdb': '1H1S', 'chain': 'A', 'group': 'CMGC', 'disease': 'Cancer'},
    {'name': 'BRAF',  'pdb': '3OG7', 'chain': 'A', 'group': 'TKL',  'disease': 'Melanoma'},
]

kinase_df = pd.DataFrame(KINASES)
kinase_df.index += 1
print(f'{len(KINASES)} kinases:')
kinase_df


### 도킹할 리간드 정의

모든 키나아제에 동일하게 도킹할 리간드의 SMILES와 이름을 입력합니다.

**설정값** (1-2. 도킹할 리간드 정의):

| 변수 | 기본값 | 타입 | 설명 |
|------|--------|------|------|
| `LIGAND_SMILES` | `C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1` | 문자열 | 리간드 SMILES 문자열 |
| `LIGAND_NAME` | `Erlotinib` | 문자열 | 리간드 이름 |


In [ ]:
#@title 1-2. 도킹할 리간드 정의 {display-mode: "form"}

LIGAND_SMILES = "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1"  #@param {type:"string"}
LIGAND_NAME = "Erlotinib"  #@param {type:"string"}

mol = Chem.MolFromSmiles(LIGAND_SMILES)
print(f'Ligand: {LIGAND_NAME}')
Draw.MolToImage(mol, size=(300, 200))


## 2. 구조 준비


### 전체 키나아제 구조 준비

각 키나아제에 대해 PDB 다운로드 → 단백질 추출 → 수소 추가 → PDBQT 변환을 수행합니다.
co-crystal 리간드의 좌표도 추출하여 docking box 계산에 사용합니다.


In [ ]:
#@title 2-1. 전체 키나아제 구조 준비 {display-mode: "form"}
from pymol import cmd as pcmd

structures = {}
pdb_infos = {}

# ========== Step 1: Fetch all PDBs ==========
for i, k in enumerate(KINASES):
    pdb_id = k['pdb'].lower()
    name = k['name']
    kdir = os.path.join(WORK_DIR, name)
    os.makedirs(kdir, exist_ok=True)
    pdb_path = os.path.join(kdir, f'{pdb_id}.pdb')
    if not os.path.exists(pdb_path):
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', pdb_path)
    pdb_infos[name] = {'pdb_id': pdb_id, 'pdb_path': pdb_path, 'dir': kdir,
                        'chain': k['chain'], 'group': k['group']}
    print(f'[{i+1}/{len(KINASES)}] {name} ({pdb_id.upper()}) fetched')

# ========== Step 2: PyMOL 정렬 (가장 먼저!) ==========
pcmd.delete('all')
kinase_names_list = list(pdb_infos.keys())
ref_name = kinase_names_list[0]
print(f'\nAligning to reference: {ref_name}')

for name, info in pdb_infos.items():
    pcmd.load(info['pdb_path'], f'{name}_all')
    pcmd.remove(f'{name}_all and not chain {info["chain"]}')
    pcmd.remove(f'{name}_all and resn HOH+WAT+SOL+GOL+PEG+EDO+PG4+DMS+ACT+BME+CL+NA+MG+ZN+CA+SO4+MN+FE+CU+CO+NI+CD+K+BR+IOD+PO4+NO3+SCN+CIT+FMT+TRS+MPD+IPA+EOH+MLI+AZI')

align_rmsds = {(ref_name, ref_name): 0.0}
for name in kinase_names_list[1:]:
    try:
        result = pcmd.super(f'{name}_all and polymer', f'{ref_name}_all and polymer')
        align_rmsds[(ref_name, name)] = round(result[0], 2)
        align_rmsds[(name, ref_name)] = round(result[0], 2)
        print(f'  {name} → {ref_name}: RMSD={result[0]:.2f} A')
    except Exception as e:
        print(f'  {name} align failed: {e}')

# Save aligned PDBs
for name in kinase_names_list:
    aligned_pdb = os.path.join(pdb_infos[name]['dir'], f'{pdb_infos[name]["pdb_id"]}_aligned.pdb')
    pcmd.save(aligned_pdb, f'{name}_all')
    pdb_infos[name]['aligned'] = aligned_pdb

pcmd.delete('all')
print('All kinases aligned.')

# ========== Step 3: 정렬된 PDB에서 처리 ==========
for name, info in pdb_infos.items():
    pdb_id = info['pdb_id']
    kdir = info['dir']
    aligned_pdb = info['aligned']
    print(f'\n[{name}] Processing...', end=' ', flush=True)

    try:
        u = mda.Universe(aligned_pdb)
        prot_sel = u.select_atoms('protein')
        clean_pdb = os.path.join(kdir, f'{pdb_id}_clean.pdb')
        prot_sel.write(clean_pdb)

        # PDBFixer (정렬된 좌표)
        prot_H = os.path.join(kdir, f'{pdb_id}_clean_H.pdb')
        fixer = PDBFixer(filename=clean_pdb)
        fixer.findMissingResidues(); fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues(); fixer.removeHeterogens(True)
        fixer.findMissingAtoms(); fixer.addMissingAtoms(); fixer.addMissingHydrogens(7.4)
        with open(prot_H, 'w') as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)

        # Receptor PDBQT (정렬된 좌표)
        rec_qt = os.path.join(kdir, f'{pdb_id}_rec.pdbqt')
        mol_ob = list(pybel.readfile(format='pdb', filename=prot_H))[0]
        out = pybel.Outputfile(filename=rec_qt+'.tmp', format='pdbqt', overwrite=True)
        out.write(mol_ob); out.close()
        with open(rec_qt+'.tmp') as f: raw = f.readlines()
        with open(rec_qt, 'w') as f:
            for l in raw:
                if not l.startswith(('ROOT','ENDROOT','BRANCH','ENDBRANCH','TORSDOF')): f.write(l)
        os.remove(rec_qt+'.tmp')

        # 리간드 좌표 (정렬된 좌표에서)
        lig_sel = u.select_atoms('not protein and not nucleic and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4 MN FE CU CO NI CD K BR IOD PO4 NO3 SCN CIT FMT TRS MPD IPA EOH MLI AZI')
        if len(lig_sel) > 0:
            resnames = list(set(lig_sel.resnames))
            if len(resnames) > 1:
                biggest = max(resnames, key=lambda r: len(lig_sel.select_atoms(f'resname {r}')))
                lig_sel = lig_sel.select_atoms(f'resname {biggest}')

        # Crystal 리간드를 SDF로 저장 (정렬된 좌표)
        lig_sdf = os.path.join(kdir, f'{name}_crystal.sdf')
        lig_mol2 = os.path.join(kdir, f'{pdb_id}_lig.mol2')
        if len(lig_sel) > 0:
            lig_pdb_tmp = os.path.join(kdir, f'{pdb_id}_lig_tmp.pdb')
            lig_sel.write(lig_pdb_tmp)
            try:
                obmol = list(pybel.readfile(format='pdb', filename=lig_pdb_tmp))[0]
                lout = pybel.Outputfile(filename=lig_mol2, format='mol2', overwrite=True)
                lout.write(obmol); lout.close()
                lout2 = pybel.Outputfile(filename=lig_sdf, format='sdf', overwrite=True)
                lout2.write(obmol); lout2.close()
                os.remove(lig_pdb_tmp)
            except:
                lig_mol2 = None; lig_sdf = None
        else:
            lig_mol2 = None; lig_sdf = None

        structures[name] = {
            'pdb_id': pdb_id, 'pdb_path': info['pdb_path'], 'dir': kdir,
            'prot_H': prot_H, 'rec_qt': rec_qt, 'group': info['group'],
            'chain': info['chain'],
            'lig_coords': lig_sel.positions if len(lig_sel) > 0 else None,
            'lig_mol2': lig_mol2,
            'lig_sdf': lig_sdf,
        }
        print('OK')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\n=== {len(structures)}/{len(KINASES)} prepared (all in aligned coords) ===')


### 구조 정렬 & RMSD 매트릭스

PyMOL CE-Align으로 키나아제 구조를 정렬합니다.
ATP binding site는 키나아제 간 보존되어 있어 정렬이 가능합니다.


In [ ]:
#@title 2.5-1. RMSD 매트릭스 {display-mode: "form"}
from pymol import cmd as pcmd

kinase_names = list(structures.keys())

# Full pairwise RMSD
pcmd.delete('all')
for name in kinase_names:
    pcmd.load(structures[name]['prot_H'], f'{name}_prot')

for i, id1 in enumerate(kinase_names):
    for j, id2 in enumerate(kinase_names):
        if i == j:
            align_rmsds[(id1, id2)] = 0.0
        elif (id1, id2) not in align_rmsds:
            try:
                pcmd.create('tmp_mob', f'{id2}_prot')
                result = pcmd.super('tmp_mob', f'{id1}_prot')
                align_rmsds[(id1, id2)] = round(result[0], 2)
                align_rmsds[(id2, id1)] = round(result[0], 2)
                pcmd.delete('tmp_mob')
            except:
                align_rmsds[(id1, id2)] = np.nan
                pcmd.delete('tmp_mob')

pcmd.delete('all')

rmsd_matrix = pd.DataFrame(
    [[align_rmsds.get((id1, id2), np.nan) for id2 in kinase_names] for id1 in kinase_names],
    index=kinase_names, columns=kinase_names
)
print('Alignment RMSD Matrix:')
print(rmsd_matrix.to_string(float_format='%.2f'))


### RMSD 히트맵 & 3D 비교


In [ ]:
#@title 2.5-2. RMSD 히트맵 & 3D 비교 {display-mode: "form"}
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(rmsd_matrix.astype(float), annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'RMSD (A)'})
ax.set_title('Kinase Panel Alignment RMSD')
plt.tight_layout()
plt.show()

view = py3Dmol.view(width=800, height=600)
k_colors = ['#43A047', '#1E88E5', '#E53935', '#FF8F00', '#8E24AA']
model_idx = 0
for i, name in enumerate(kinase_names):
    prot = structures[name].get('prot_H_aligned', structures[name]['prot_H'])
    with open(prot) as f:
        view.addModel(f.read(), 'pdb')
    view.setStyle({'model': model_idx}, {'cartoon': {'color': k_colors[i % len(k_colors)], 'opacity': 0.7}})
    model_idx += 1
print(', '.join([f'{n}={k_colors[i%len(k_colors)]}' for i, n in enumerate(kinase_names)]))
view.zoomTo()
view.show()


## 3. Docking Box 설정


### Docking Box 설정

각 키나아제의 co-crystal 리간드 위치를 기준으로 개별 docking box를 생성합니다.
서로 다른 키나아제이므로 각자의 결합 부위에 맞는 box가 필요합니다.

**설정값** (3-1. Box 설정 (정렬된 좌표 기준)):

| 변수 | 기본값 | 타입 | 설명 |
|------|--------|------|------|
| `BOX_METHOD` | `auto` | 선택: auto/residue/manual | 도킹 박스 설정 방식 |
| `RESIDUE_LIST` | `790, 858, 855` | 문자열 | 박스 중심 잔기 번호 (쉼표 구분) |
| `MANUAL_CENTER` | `12.5, -3.0, 27.0` | 문자열 | 박스 중심 좌표 (x, y, z) |
| `MANUAL_SIZE` | `25.0, 25.0, 25.0` | 문자열 | 박스 크기 (x, y, z) Å |
| `PADDING` | `7.0` | 숫자 | 도킹 박스 여유 공간 (Å) |


In [ ]:
#@title 3-1. Box 설정 (정렬된 좌표 기준) {display-mode: "form"}
BOX_METHOD = "auto"  #@param ["auto", "residue", "manual"]
RESIDUE_LIST = "790, 858, 855"  #@param {type:"string"}
MANUAL_CENTER = "12.5, -3.0, 27.0"  #@param {type:"string"}
MANUAL_SIZE = "25.0, 25.0, 25.0"  #@param {type:"string"}
PADDING = 7.0  #@param {type:"number"}

def get_box_from_coords(coords, padding=7.0):
    minC, maxC = coords.min(axis=0), coords.max(axis=0)
    return (
        {'x': float((maxC[0]+minC[0])/2), 'y': float((maxC[1]+minC[1])/2), 'z': float((maxC[2]+minC[2])/2)},
        {'x': float(maxC[0]-minC[0]+2*padding), 'y': float(maxC[1]-minC[1]+2*padding), 'z': float(maxC[2]-minC[2]+2*padding)}
    )

# 정렬 후에는 모든 구조가 reference 좌표계에 있으므로
# reference 구조의 리간드 위치로 공통 box 사용
ref = list(structures.keys())[0]

if BOX_METHOD == "auto":
    # Reference의 리간드 좌표 사용 (정렬 기준)
    if structures[ref]['lig_coords'] is not None:
        center, size = get_box_from_coords(structures[ref]['lig_coords'], PADDING)
        print(f'Auto box from reference ({ref}) ligand')
    else:
        print('ERROR: reference has no ligand coordinates')
elif BOX_METHOD == "residue":
    res_nums = [int(r.strip()) for r in RESIDUE_LIST.split(',')]
    ref_prot = mda.Universe(structures[ref]['prot_H'])
    sel = ref_prot.select_atoms(' or '.join([f'resid {r}' for r in res_nums]))
    if len(sel) > 0:
        center, size = get_box_from_coords(sel.positions, PADDING)
    print(f'Residue box from {res_nums}')
elif BOX_METHOD == "manual":
    cx,cy,cz = [float(v) for v in MANUAL_CENTER.split(',')]
    sx,sy,sz = [float(v) for v in MANUAL_SIZE.split(',')]
    center, size = {'x':cx,'y':cy,'z':cz}, {'x':sx,'y':sy,'z':sz}
    print('Manual box')

# 모든 키나아제에 동일한 box 적용
for name, s in structures.items():
    s['center'] = center
    s['size'] = size

print(f'Center: ({center["x"]:.1f}, {center["y"]:.1f}, {center["z"]:.1f})')
print(f'Size:   ({size["x"]:.1f}, {size["y"]:.1f}, {size["z"]:.1f})')
print(f'→ 정렬된 모든 키나아제에 동일한 box 적용')

### Box 3D 시각화

첫 번째 키나아제의 docking box를 3D로 확인합니다.


In [ ]:
#@title 3-2. Box 3D 시각화 (첫 번째 키나아제) {display-mode: "form"}
first_name = list(structures.keys())[0]
s = structures[first_name]

view = py3Dmol.view(width=800, height=600)
with open(s['prot_H']) as f: view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.6}})
view.addBox({
    'center': {'x': s['center']['x'], 'y': s['center']['y'], 'z': s['center']['z']},
    'dimensions': {'w': s['size']['x'], 'h': s['size']['y'], 'd': s['size']['z']},
    'color': 'blue', 'opacity': 0.15
})
print(f'{first_name}: Box 시각화')
view.zoomTo()
view.show()


## 4. Cross-docking

**설정값** (4-1. 리간드 PDBQT 준비 + 전체 도킹):

| 변수 | 기본값 | 타입 | 설명 |
|------|--------|------|------|
| `EXHAUSTIVENESS` | `16` | 정수 | 도킹 탐색 깊이 (높을수록 정확, 느림) |
| `N_POSES` | `5` | 정수 | 생성할 도킹 포즈 수 |
| `N_CPUS` | `10` | 정수 | CPU 코어 수 (0=자동) |


In [ ]:
#@title 4-1. 리간드 PDBQT 준비 + 전체 도킹 {display-mode: "form"}
EXHAUSTIVENESS = 16  #@param {type:"integer"}
N_POSES = 5          #@param {type:"integer"}
N_CPUS = 10          #@param {type:"integer"}

smina = os.path.join(BIN_DIR, 'smina')

# Prepare ligand PDBQT
rdmol = Chem.MolFromSmiles(LIGAND_SMILES)
rdmol = Chem.AddHs(rdmol)
AllChem.EmbedMolecule(rdmol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(rdmol)

lig_sdf = os.path.join(WORK_DIR, f'{LIGAND_NAME}.sdf')
writer = Chem.SDWriter(lig_sdf)
writer.write(rdmol)
writer.close()

# Dock against each kinase
t0 = time.time()
results = []

for i, (name, s) in enumerate(structures.items()):
    print(f'[{i+1}/{len(structures)}] {name}...', end=' ', flush=True)
    
    sdf_out = os.path.join(s['dir'], f'{LIGAND_NAME}_docked.sdf')
    subprocess.run([
        smina, '-r', s['rec_qt'], '-l', lig_sdf, '-o', sdf_out,
        '--center_x', str(s['center']['x']), '--center_y', str(s['center']['y']), '--center_z', str(s['center']['z']),
        '--size_x', str(s['size']['x']), '--size_y', str(s['size']['y']), '--size_z', str(s['size']['z']),
        '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(N_POSES),
        '--cpu', str(N_CPUS),
    ], stdout=None, stderr=None)
    
    score = None
    if os.path.exists(sdf_out) and os.path.getsize(sdf_out) > 0:
        suppl = list(Chem.SDMolSupplier(sdf_out, removeHs=False))
        scores = []
        for pose_mol in suppl:
            if pose_mol is None: continue
            p = pose_mol.GetPropsAsDict()
            if 'minimizedAffinity' in p:
                try: scores.append(float(p['minimizedAffinity']))
                except: pass
        if scores:
            score = min(scores)
    
    results.append({
        'Kinase': name, 'Group': s['group'],
        'Score': round(score, 2) if score else None,
        'SDF': sdf_out,
    })
    print(f'{score:.2f}' if score else 'FAILED')

elapsed = time.time() - t0
print(f'\n=== Done in {elapsed:.0f}s ===')


### Crystal 포즈 스코어링

각 구조의 co-crystal 리간드를 원래 위치에서 `--score_only`로 스코어링합니다.
도킹 포즈의 스코어와 비교하여 scoring function의 신뢰성을 확인합니다.


In [ ]:
#@title 4-2. Crystal 포즈 스코어링 {display-mode: "form"}
print('Crystal ligand scoring (aligned coords)...', flush=True)

for name, s in structures.items():
    crystal_sdf = os.path.join(s['dir'], f'{name}_crystal.sdf')
    lig_path = s.get('lig_mol2')
    
    if not lig_path or not os.path.exists(str(lig_path)):
        print(f'{name}: no aligned ligand')
        s['crystal_score'] = None
        continue
    
    try:
        obmol = list(pybel.readfile(format='mol2', filename=lig_path))[0]
        lout = pybel.Outputfile(filename=crystal_sdf, format='sdf', overwrite=True)
        lout.write(obmol); lout.close()
    except Exception as e:
        print(f'{name}: crystal SDF failed - {e}')
        s['crystal_score'] = None
        continue
    
    rec_qt = os.path.join(s['dir'], f'{s["pdb_id"]}_rec.pdbqt')
    score_out = os.path.join(s['dir'], f'{name}_crystal_scored.sdf')
    os.system(f'{smina} -r {rec_qt} -l {crystal_sdf} -o {score_out} --score_only 2>/dev/null')
    
    crystal_score = None
    if os.path.exists(score_out) and os.path.getsize(score_out) > 0:
        suppl = list(Chem.SDMolSupplier(score_out, removeHs=False))
        if suppl and suppl[0]:
            props = suppl[0].GetPropsAsDict()
            if 'minimizedAffinity' in props:
                try: crystal_score = float(props['minimizedAffinity'])
                except: pass
    
    s['crystal_score'] = crystal_score
    if crystal_score is not None:
        print(f'{name}: crystal={crystal_score:.2f} kcal/mol')
    else:
        print(f'{name}: scoring failed')

## 5. 선택성 분석


### 선택성 프로파일

스코어별로 키나아제를 정렬하여 리간드가 어떤 키나아제에 가장 잘 결합하는지 확인합니다.
- **Selectivity Index**: 최적 스코어 대비 차이. 0에 가까울수록 선택적


In [ ]:
#@title 5-1. 스코어 비교 테이블 & 차트 {display-mode: "form"}
df = pd.DataFrame(results).dropna(subset=['Score']).sort_values('Score')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score by kinase
group_colors = {'TK': '#E74C3C', 'CMGC': '#3498DB', 'TKL': '#2ECC71', 'AGC': '#F39C12', 'Other': '#9B59B6'}
colors = [group_colors.get(g, 'gray') for g in df['Group']]
axes[0].barh(df['Kinase'], df['Score'], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Docking Score (kcal/mol)')
axes[0].set_title(f'{LIGAND_NAME} Selectivity Profile')
axes[0].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5)
for _, row in df.iterrows():
    axes[0].text(row['Score']-0.1, row['Kinase'], f'{row["Score"]:.1f}', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

# Score by group
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=g) for g, c in group_colors.items() if g in df['Group'].values]
axes[0].legend(handles=legend_elements, loc='lower right')

# Selectivity index
best = df['Score'].min()
df['Selectivity'] = df['Score'] - best
axes[1].barh(df['Kinase'], df['Selectivity'], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('ΔScore from best (kcal/mol)')
axes[1].set_title('Selectivity Index (0 = most selective)')

plt.tight_layout()
plt.show()

print(f'\nBest binding: {df.iloc[0]["Kinase"]} ({df.iloc[0]["Score"]:.2f} kcal/mol)')
df[['Kinase', 'Group', 'Score', 'Selectivity']]


### ProLIF 상호작용 비교

각 키나아제에서의 상호작용 유형(수소결합, 소수성 등)을 비교합니다.
선택성의 구조적 근거를 파악할 수 있습니다.


In [ ]:
#@title 5-2. ProLIF 상호작용 비교 {display-mode: "form"}
interaction_data = []

for _, row in df.iterrows():
    name = row['Kinase']
    s = structures[name]
    sdf_out = row['SDF']
    if not os.path.exists(sdf_out): continue
    
    try:
        prot_mol = Chem.MolFromPDBFile(s['prot_H'], removeHs=False, sanitize=False)
        Chem.SanitizeMol(prot_mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
        prot_plf = plf.Molecule(prot_mol)
        
        suppl = Chem.SDMolSupplier(sdf_out, sanitize=False, removeHs=False)
        ligs = []
        for mol in suppl:
            if mol is None: continue
            try:
                Chem.SanitizeMol(mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
                ligs.append(plf.Molecule(mol))
            except: pass
        
        if ligs:
            fp = plf.Fingerprint()
            fp.run_from_iterable(ligs, prot_plf, n_jobs=1)
            df_ifp = fp.to_dataframe()
            interactions = {}
            for col in df_ifp.columns:
                itype = str(col[2]) if isinstance(col, tuple) and len(col) > 2 else str(col)
                if df_ifp[col].any():
                    interactions[itype] = interactions.get(itype, 0) + 1
            interaction_data.append({
                'Kinase': name, 'Score': row['Score'],
                'HBond': interactions.get('HBDonor', 0) + interactions.get('HBAcceptor', 0),
                'Hydrophobic': interactions.get('Hydrophobic', 0),
                'Total': sum(interactions.values()),
            })
    except Exception as e:
        print(f'{name}: ProLIF failed - {e}')

if interaction_data:
    int_df = pd.DataFrame(interaction_data)
    print('Interaction comparison:')
    int_df


### 포즈 3D 비교

각 키나아제에서의 도킹 포즈를 겹쳐서 3D로 비교합니다.


In [ ]:
#@title 5-3. 포즈 3D 비교 {display-mode: "form"}
view = py3Dmol.view(width=800, height=600)

colors = ['#43A047', '#1E88E5', '#E53935', '#FF8F00', '#8E24AA']
model_idx = 0

for i, (name, s) in enumerate(structures.items()):
    sdf_out = os.path.join(s['dir'], f'{LIGAND_NAME}_docked.sdf')
    if not os.path.exists(sdf_out): continue
    suppl = Chem.SDMolSupplier(sdf_out, removeHs=False, sanitize=False)
    if suppl and suppl[0]:
        block = Chem.MolToMolBlock(suppl[0])
        view.addModel(block, 'mol')
        view.setStyle({'model': model_idx}, {'stick': {'color': colors[i % len(colors)], 'radius': 0.2}})
        model_idx += 1

legend = ', '.join([f'{name}={colors[i%len(colors)]}' for i, name in enumerate(structures.keys())])
print(f'{LIGAND_NAME} in: {legend}')
view.zoomTo()
view.show()


## 6. 내보내기


### 결과 내보내기

CSV + PyMOL 스크립트 + ZIP 파일로 저장합니다.


In [ ]:
#@title 6-1. 결과 내보내기 {display-mode: "form"}
import shutil

csv_path = os.path.join(WORK_DIR, 'kinase_panel_results.csv')
df.to_csv(csv_path, index=False)
print(f'CSV: {csv_path}')

pml_path = os.path.join(WORK_DIR, 'kinase_panel_pymol.pml')
with open(pml_path, 'w') as f:
    f.write('reinitialize\nbg_color white\n\n')
    pml_colors = ['palegreen','lightblue','salmon','lightorange','lightpink']
    for i, (name, s) in enumerate(structures.items()):
        f.write(f'load {s["prot_H"]}, {name}\ncolor {pml_colors[i%len(pml_colors)]}, {name}\nshow cartoon, {name}\n')
        sdf = os.path.join(s['dir'], f'{LIGAND_NAME}_docked.sdf')
        if os.path.exists(sdf):
            f.write(f'load {sdf}, {name}_dock\nset all_states, 0\nframe 1\nshow sticks, {name}_dock\n\n')
print(f'PyMOL: {pml_path}')

zip_path = os.path.join(os.path.dirname(WORK_DIR), 'kinase_panel_results')

# 단백질 PDB + 도킹 포즈를 뷰어용 폴더에 복사
import glob as _glob
results_dir = os.path.join(WORK_DIR, 'results_for_viewer')
os.makedirs(results_dir, exist_ok=True)
for _name, _s in structures.items():
    _prot = _s['prot_H']
    if os.path.exists(_prot): shutil.copy2(_prot, os.path.join(results_dir, f'{_name}_protein.pdb'))
for _sdf in _glob.glob(os.path.join(WORK_DIR, '**/*_docked.sdf'), recursive=True):
    shutil.copy2(_sdf, results_dir)
print(f'뷰어용 파일: {results_dir}/')
print('  PyMOL/ChimeraX: protein.pdb + *_docked.sdf 를 함께 열기')

shutil.make_archive(zip_path, 'zip', WORK_DIR)
print(f'Archive: {zip_path}.zip')
try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Results at: {WORK_DIR}')
